# 01. Baseline

Comparison of Classic vs. Classic+hint. Motivation for the neural network approach.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from reedsolo import RSCodec, ReedSolomonError

sys.path.append('..')
from rs.channels import qsc_erasure_channel

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parents[0]
table_out_dir = ROOT / "tables"
table_out_dir.mkdir(parents=True, exist_ok=True)

graph_out_dir = ROOT / "graphs"
graph_out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
N, K, NSYM = 255, 223, 32
rsc = RSCodec(NSYM)

In [ ]:
def evaluate_baseline(p_err, p_erase, num_samples=1000):
    classic_ok, hint_ok = 0, 0
    for _ in range(num_samples):
        msg = os.urandom(K)
        codeword = rsc.encode(msg)
        noisy, erase_pos = qsc_erasure_channel(codeword, p_err, p_erase)
        
        try:
            if bytes(rsc.decode(noisy)[0]) == msg:
                classic_ok += 1
        except: pass
        
        try:
            if bytes(rsc.decode(noisy, erase_pos=erase_pos)[0]) == msg:
                hint_ok += 1
        except: pass
    
    return classic_ok / num_samples, hint_ok / num_samples

In [ ]:
P_ERR = 0.02
p_erase_values = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14]

results = []
for p_erase in p_erase_values:
    classic, hint = evaluate_baseline(P_ERR, p_erase)
    results.append({'p_erase': p_erase, 'classic': classic, 'hint': hint})
    print(f'p_erase={p_erase}: Classic={classic:.1%}, Hint={hint:.1%}')

df = pd.DataFrame(results, index=False)

df.to_csv(table_out_dir / "gap.csv", index = False)
print(df.to_string())

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot([r['p_erase'] for r in results], [r['hint'] for r in results], 'g-o', label='Classic+hint')
plt.plot([r['p_erase'] for r in results], [r['classic'] for r in results], 'r-s', label='Classic')
plt.fill_between([r['p_erase'] for r in results], [r['classic'] for r in results], [r['hint'] for r in results], alpha=0.3)
plt.xlabel('p_erase'); plt.ylabel('FSR'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Gap между Classic и Classic+hint')

plt.savefig(graph_out_dir / 'gap.png', dpi=150); plt.show()